
## 1. Define Activation Logic

Using your half-hourly temperatures, secondary heating can be modeled using a few **rules**:

1. **Temperature threshold (`T_cold`)**

   * Decide below which outdoor temperature secondary heating is likely used.
   * Example for rural Scotland: `T_cold = 8 °C` (can tune later).

2. **Occupancy weighting**

   * Secondary heating is mostly used when people are home:

     * Morning: 06:00–09:00
     * Evening: 17:00–22:00

3. **Maximum secondary heat fraction (`f_max`)**

   * Wood stove: 20–50 % of instantaneous demand
   * Electric heater: 10–30 %

---

## 2. Compute Secondary Heating Fraction

For each half hour:

[
f_{\text{secondary}}(t) =
\begin{cases}
f_{\text{max}} & \text{if } T_{\text{out}} < T_{\text{cold}} \text{ and } t \in \text{occupied hours} \
0 & \text{otherwise}
\end{cases}
]

Then assign:

[
Q_{\text{secondary}}(t) = f_{\text{secondary}}(t) \cdot Q_{\text{demand}}(t)
]

[
Q_{\text{primary}}(t) = Q_{\text{demand}}(t) - Q_{\text{secondary}}(t)
]

---

## 3. Optional: Smooth or Probabilistic Activation

* Wood stoves are intermittent. To reflect that:

  * Introduce a **probabilistic switch**: 50–70 % chance of secondary heating being on when conditions are met.
  * Helps avoid unrealistic continuous use.

* You could also **scale linearly with temperature**:
  [
  f_{\text{secondary}}(t) = f_{\text{max}} \cdot \frac{T_{\text{cold}} - T_{\text{out}}(t)}{T_{\text{cold}} - T_{\text{min}}}
  ]

  * Where `T_min` is the coldest observed temperature, ensuring maximum fraction when very cold.

---

## 4. Implementation in Python

Here’s a compact workflow:

```python
import pandas as pd

# Load CSV
df = pd.read_csv("ambient_temp.csv", sep="\t")  # adjust separator if needed

# Add time index
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M')
df.set_index('Time', inplace=True)

# Parameters
T_cold = 8        # °C, threshold for secondary heating
f_max = 0.3       # max fraction of demand met by secondary heating
occupied_hours = range(6, 10)  # morning
occupied_hours = list(occupied_hours) + list(range(17, 23))  # add evening

# Example ESP-r half-hourly demand series (kW)
df['Q_demand'] = 2.0  # placeholder, replace with actual ESP-r data

# Determine secondary heating fraction
df['f_secondary'] = 0.0
for i, row in df.iterrows():
    hour = i.hour
    if row['Ambientdb (¬∞C)'] < T_cold and hour in occupied_hours:
        df.at[i, 'f_secondary'] = f_max

# Compute energy split
df['Q_secondary'] = df['f_secondary'] * df['Q_demand']
df['Q_primary'] = df['Q_demand'] - df['Q_secondary']

print(df.head(20))
```

* `Q_secondary` is your **half-hourly secondary heat use**
* `Q_primary` is what the main heating provides
* This method is fully adjustable: thresholds, f_max, occupancy windows



In [1]:
# Step 1: Import Libraries
# ------------------------
# Description: Import all necessary Python libraries for data manipulation and datetime handling.

import pandas as pd
import numpy as np


In [2]:
import pandas as pd

temp_file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/cleaned_OAtemp.csv"
df_temp = pd.read_csv(temp_file_path)

# Show the column names exactly as Python sees them
print(df_temp.columns.tolist())

['Time', 'Ambientdb (°C)']


In [3]:
# Step 2: Load Air Temperature Data
# ---------------------------------
# Description: Load your half-hourly outdoor air temperature CSV. The CSV has columns:
# 'Time' (HH:MM format) and 'Ambientdb (¬∞C)' for temperature in Celsius.
# File path is set as per your instruction. Adjust separator if necessary.

# Parameters
temp_file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/cleaned_OAtemp.csv"

# Load CSV
df_temp = pd.read_csv(temp_file_path)

# Convert 'Time' to datetime index for 2026
# We will generate a full datetime for 2026 using the first column for HH:MM
# Assuming the CSV is ordered consecutively through the year
df_temp['DateTime'] = pd.date_range(start='2026-01-01 00:30', periods=len(df_temp), freq='30T')
df_temp.set_index('DateTime', inplace=True)

# Keep only the temperature column and rename for clarity
df_temp = df_temp[['Ambientdb (°C)']].rename(columns={'Ambientdb (°C)': 'T_out'})

/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_28837/2532442737.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_temp['DateTime'] = pd.date_range(start='2026-01-01 00:30', periods=len(df_temp), freq='30T')


In [4]:
# Step 3: Define Parameters for Secondary Heating
# -----------------------------------------------
# Description: Define all adjustable parameters for the secondary heating logic. 
# This includes temperature threshold, maximum fraction of demand, and occupancy windows.

# Temperature threshold below which secondary heating is used (°C)
T_cold = 8

# Maximum proportion of total heating demand that secondary heating can meet
f_max = 0.3  # 30% of demand

# Occupancy schedules (hour ranges) in 24-hour format
# Weekdays: Morning 06:00-09:00, Evening 17:00-22:00
weekday_morning_hours = range(6, 10)
weekday_evening_hours = range(17, 23)

# Weekends: Morning 07:00-11:00, Evening 16:00-23:00
weekend_morning_hours = range(7, 12)
weekend_evening_hours = range(16, 24)

# Optional: minimum demand fraction below which secondary heating is never triggered
min_fraction_demand = 0.05  # 5% (can be adjusted)

In [5]:
# Step 4: Determine Occupancy for Each Timestamp
# ----------------------------------------------
# Description: Assign occupancy based on weekday/weekend and the hour of the day. 
# Occupancy will affect whether secondary heating is activated.

# Function to determine occupancy hours
def is_occupied(timestamp):
    if timestamp.weekday() < 5:  # Weekday (Mon-Fri)
        return (timestamp.hour in weekday_morning_hours) or (timestamp.hour in weekday_evening_hours)
    else:  # Weekend (Sat-Sun)
        return (timestamp.hour in weekend_morning_hours) or (timestamp.hour in weekend_evening_hours)

# Apply function to DataFrame
df_temp['Occupied'] = df_temp.index.map(is_occupied)

In [6]:
# Step 5: Compute Secondary Heating Activation
# --------------------------------------------
# Description: Apply simple rules to determine secondary heating usage:
# If the temperature is below T_cold and the house is occupied, secondary heating can activate.

# Activation is binary: 1 if active, 0 if not
df_temp['Secondary_Activation'] = np.where((df_temp['T_out'] < T_cold) & (df_temp['Occupied']), 1, 0)

In [7]:
# Step 6: Compute Secondary Heating Percent
# -----------------------------------------
# Description: Determine the fraction of heating demand met by secondary heating. 
# Formula used:
#   Q_secondary_fraction = f_max if activated, else 0

df_temp['Secondary_Heating_Percent'] = df_temp['Secondary_Activation'] * f_max

In [8]:
# Step 6b: Compute Primary Heating Percent
# ----------------------------------------
# Description: Determine the fraction of heating demand met by the primary heating system.
# Formula used:
#   Primary_Heating_Percent = 1 - Secondary_Heating_Percent

df_temp['Primary_Heating_Percent'] = 1 - df_temp['Secondary_Heating_Percent']

# Optional: check the first few rows
print(df_temp[['Secondary_Heating_Percent', 'Primary_Heating_Percent']].head(10))

                     Secondary_Heating_Percent  Primary_Heating_Percent
DateTime                                                               
2026-01-01 00:30:00                        0.0                      1.0
2026-01-01 01:00:00                        0.0                      1.0
2026-01-01 01:30:00                        0.0                      1.0
2026-01-01 02:00:00                        0.0                      1.0
2026-01-01 02:30:00                        0.0                      1.0
2026-01-01 03:00:00                        0.0                      1.0
2026-01-01 03:30:00                        0.0                      1.0
2026-01-01 04:00:00                        0.0                      1.0
2026-01-01 04:30:00                        0.0                      1.0
2026-01-01 05:00:00                        0.0                      1.0


In [9]:
# Step 7: Clean Data and Prepare Output
# -------------------------------------
# Description: Keep only the relevant columns for output CSV: Time, Secondary Activation, Secondary Heating Percent.


df_output = df_temp[['Secondary_Activation', 'Secondary_Heating_Percent', 'Primary_Heating_Percent']].copy()
df_output.insert(0, 'Time', df_output.index.strftime('%H:%M'))

In [10]:
# Step 8: Save to CSV
# -------------------
# Description: Save the half-hourly secondary heating CSV for the full year 2026.

output_file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/secondary_heating_activation.csv"
df_output.to_csv(output_file_path, index=False)

print(f"CSV saved: {output_file_path}")

CSV saved: /Users/jakegehrung/Desktop/data_raw/data_processed/secondary_heating_activation.csv
